# 04 · Reranker로 결정 추천 만들기

**목표**: 고객 행동 원문과 YAML 정책 후보를 rerank하여 `approve`, `manual-review`, `decline` 중 하나를 **추천**합니다. 기본 V3는 입력 원문·확인 가능한 사실과 균형 잡힌 후보 원형을 함께 비교해 관련도 점수로 순위를 정합니다.

이 실습은 실제 승인·반려를 실행하지 않습니다. 기본 `min_score`, `min_margin`은 모두 `0.0`이므로 점수·margin 임계값으로 결과를 바꾸지 않습니다. `manual-review`는 모델이 해당 YAML 후보를 1위로 선택했을 때의 추천 결과입니다. 실행 전 루트 `.env`에 배포된 `RERANK_MODEL`을 설정하세요. LiteLLM/Cohere 호환 endpoint를 쓸 때만 `RERANK_ENDPOINT`, `RERANK_API_KEY`를 추가합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 정책 후보와 비교하기 (UI)</h2>
<p style="margin: 7px 0;">고객 행동을 입력하고 <b>추천 생성</b>을 누르세요. 입력 원문과 확인 가능한 수치·상태를 Cohere reranker의 query로 함께 전달하며, 결과에는 YAML 정책 후보의 순위와 score·margin이 표시됩니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>고객 행동을 자연어로 입력합니다.</li><li>예시 시나리오 카드에서 사례를 선택하거나 직접 수정합니다.</li><li>후보별 관련도와 1위 추천을 확인합니다. 기본 임계값은 모두 <code>0.0</code>입니다.</li></ol>
<p style="margin: 7px 0;">UI는 추천과 근거 조회만 수행합니다. 승인·반려·검토 큐 등록 같은 외부 업무 상태는 변경하지 않습니다.</p>
  </div>
</div>

In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

import importlib
import src.notebook_ui as notebook_ui
import src.reranker.lab_core as decision_core

importlib.reload(decision_core)
importlib.reload(notebook_ui)

from src.notebook_ui import build_decision_reranker_ui
from src.reranker.lab_core import (
    MULTI_PROTOTYPE_DECISIONS, DEFAULT_MIN_MARGIN, DEFAULT_MIN_SCORE,
    run_decision_recommendation,
)

candidate_options = [
    {"id": candidate.id, "label": candidate.label, "description": candidate.description}
    for candidate in MULTI_PROTOTYPE_DECISIONS
]
decision_ui = build_decision_reranker_ui(
    run_decision_recommendation,
    candidates=candidate_options,
    default_min_margin=DEFAULT_MIN_MARGIN,
    default_min_score=DEFAULT_MIN_SCORE,
)
decision_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", width="100%", height=900)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>추천 결과에 원문·사실 query와 YAML 정책 후보 순위가 보인다.</li><li>기본값에서 최소 score·margin은 모두 <code>0.0</code>임을 확인한다.</li><li>UI가 추천만 생성하며 실제 업무 상태를 변경하지 않는다는 것을 확인한다.</li></ul>
<p style="margin: 7px 0;">문제가 없으면 다음으로 진행하세요. 아래 코드는 같은 코어 API와 실제 요청 payload를 Python으로 살펴보는 선택 심화 자료입니다.</p>
  </div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 테스트된 코어 API로 추천 근거 읽기</h2>
<p style="margin: 7px 0;">아래 셀은 UI와 같은 <code>run_decision_recommendation()</code>을 호출합니다. 기본 V3는 고객 행동 원문·확인 가능한 사실을 Cohere reranker에 한 번 전달하고, 등수별로 균형 잡힌 후보 원형을 비교합니다. 사실 추출은 후보를 고르는 규칙이나 게이트가 아닙니다.</p>
  </div>
</div>

In [3]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
for module_path in (project_root / "1_gen_ai_hub" / "src", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.config import load_settings
from src.reranker.lab_core import (
    MULTI_PROTOTYPE_DECISIONS,
    build_rerank_payload, build_reranker_query,
    run_decision_recommendation,
)
from src.reranker.demo_cases import ORDER_RISK_DEMO_CASES

settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 실제 reranker payload 확인하기</h2>
<p style="margin: 7px 0;">다음 셀은 네트워크를 호출하지 않고, Cohere에 전달할 요청 본문을 그대로 만듭니다. <code>query</code>에 입력 원문과 추출된 사실이 함께 있는지, <code>documents</code>가 등급별 두 개씩 여섯 YAML 원형인지 확인하세요.</p>
  </div>
</div>

In [4]:
case = "인증된 고객이 주문 간격 60초 이상을 지키며 1분에 4111회 주문을 시도했다. 중복, 실패, 한도 초과는 없다."
query, _ = build_reranker_query(case)
payload = build_rerank_payload(query, MULTI_PROTOTYPE_DECISIONS, settings.rerank_model)
payload

{'model': 'cohere-rerank-pro',
 'query': 'task: customer_order_behavior_classification\nobservations:\n  authenticated: true\n  min_order_interval_seconds: 60 seconds\n  orders_per_minute: 4111 per_minute\n  duplicate_order_count: 0\n  failed_order_count: 0\n  limit_exceeded: false\n  limit_avoidance_detected: unknown\n  automation_detected: unknown\n  account_abuse_detected: unknown\ncustomer_statement: |\n  인증된 고객이 주문 간격 60초 이상을 지키며 1분에 4111회 주문을 시도했다. 중복, 실패, 한도 초과는 없다.',
 'documents': ['candidate_id: approve\nprototype: stable_low_frequency\nauthenticated: true\norders_per_minute: at_most_3\nduplicate_order_count: 0\nfailed_order_count: 0\nlimit_exceeded: false\nmatch_note: 분당 0~3회의 낮은 빈도인 경우에만 정상 후보를 선택한다.',
  'candidate_id: approve\nprototype: ordinary_repeat_purchase\nauthenticated: true\nmin_order_interval_seconds: at_least_20\norders_per_minute: at_most_3\naccount_payment_delivery: consistent\nmatch_note: 주문 빈도가 분당 3회 이하이고 정상 흐름이 확인된 경우다.',
  'candidate_id: manual-review\nprot

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 공개 API로 한 번 추천하기</h2>
<p style="margin: 7px 0;">같은 원문을 공개 API에 전달해 실제 Cohere reranker 결과를 확인합니다. 기본 score·margin은 각각 <code>0.0</code>이며, 반환된 점수는 확률이나 사실 검증 결과가 아니라 후보 문서와의 관련도입니다.</p>
  </div>
</div>

In [5]:
recommendation = run_decision_recommendation(case, settings=settings)
print(f"추천: {recommendation['recommendation_label']} ({recommendation['recommendation']})")
print(f"Reranker query: {recommendation['policy_summary']}")
print(f"최소 score / margin: {recommendation['decision_thresholds']['min_score']} / {recommendation['decision_thresholds']['min_margin']}")
print(f"margin: {recommendation['margin']:.3f}")
for candidate in recommendation["ranked_candidates"]:
    print(f"{candidate['rank']}. {candidate['label']}: {candidate['score']:.3f}")
print(recommendation["action"])

추천: 모니터링 필요 (manual-review)
Reranker query: task: customer_order_behavior_classification
observations:
  authenticated: true
  min_order_interval_seconds: 60 seconds
  orders_per_minute: 4111 per_minute
  duplicate_order_count: 0
  failed_order_count: 0
  limit_exceeded: false
  limit_avoidance_detected: unknown
  automation_detected: unknown
  account_abuse_detected: unknown
customer_statement: |
  인증된 고객이 주문 간격 60초 이상을 지키며 1분에 4111회 주문을 시도했다. 중복, 실패, 한도 초과는 없다.
최소 score / margin: 0.0 / 0.0
margin: 0.013
1. 모니터링 필요: 0.927
2. 고위험 고객: 0.914
3. 정상 고객: 0.857
추천만 생성했습니다. 승인·반려 상태는 변경하지 않았습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 예시 시나리오 전체를 라벨 평가하기</h2>
<p style="margin: 7px 0;">UI와 같은 12개 시나리오를 동일한 공개 API로 평가합니다. 주문 빈도·집중 구매, 계정·기기·네트워크 연계, 결제 이상, 배송지·수취인·연락처, 인증·행동 불일치의 다섯 조건군을 포함합니다. 기대값은 평가용 정답 라벨일 뿐, reranker 입력이나 최종 추천을 바꾸지 않습니다.</p>
  </div>
</div>

In [6]:
labeled_cases = ORDER_RISK_DEMO_CASES

evaluation_rows = []
for demo_case in labeled_cases:
    outcome = run_decision_recommendation(demo_case["case"], settings=settings)
    evaluation_rows.append({
        "scenario": demo_case["name"], "signal_group": demo_case["signal_group"],
        "expected": demo_case["expected"], "predicted": outcome["recommendation"],
        "correct": outcome["recommendation"] == demo_case["expected"],
        "top_score": outcome["ranked_candidates"][0]["score"], "margin": outcome["margin"],
    })

evaluation_rows

[{'scenario': '정상 · 기본 주문',
  'signal_group': 'normal_flow',
  'expected': 'approve',
  'predicted': 'approve',
  'correct': True,
  'top_score': 0.89959866,
  'margin': 0.007373689999999988},
 {'scenario': '정상 · 안정적 반복 구매',
  'signal_group': 'normal_flow',
  'expected': 'approve',
  'predicted': 'approve',
  'correct': True,
  'top_score': 0.92566115,
  'margin': 0.017742990000000014},
 {'scenario': '관찰 · 빈번한 주문',
  'signal_group': 'velocity',
  'expected': 'manual-review',
  'predicted': 'manual-review',
  'correct': True,
  'top_score': 0.907632,
  'margin': 0.004902249999999997},
 {'scenario': '관찰 · 기기·배송지 연계',
  'signal_group': 'account_device_delivery_linkage',
  'expected': 'manual-review',
  'predicted': 'decline',
  'correct': False,
  'top_score': 0.8936255,
  'margin': 0.006773240000000014},
 {'scenario': '관찰 · 결제수단 변경',
  'signal_group': 'payment_irregularity',
  'expected': 'manual-review',
  'predicted': 'decline',
  'correct': False,
  'top_score': 0.8961969,
  'margin':

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">중간 체크포인트</h2>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>정확도는 기대 라벨과 모델 추천의 일치 비율이다.</li><li>모든 요청은 원문·사실 query와 YAML 후보를 사용한다.</li><li>score는 확률이 아니며, 이 작은 예시 평가셋의 정확도를 운영 성능으로 일반화해서는 안 된다.</li></ol>
  </div>
</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">예시 평가 정확도 계산하기</h2>
<p style="margin: 7px 0;">공개 API가 반환한 결과만 이용해 정확도를 계산합니다. 기본 경로에는 규칙 기반 재분류나 정책 override가 없습니다.</p>
  </div>
</div>


In [7]:
correct_count = sum(row["correct"] for row in evaluation_rows)
accuracy = correct_count / len(evaluation_rows)

for row in evaluation_rows:
    status = "✓" if row["correct"] else "✗"
    print(f"{status} [{row['signal_group']}] {row['scenario']}: 기대={row['expected']}, 추천={row['predicted']} | score={row['top_score']:.3f}, margin={row['margin']:.3f}")
print()
print(f"예시 시나리오 정확도: {correct_count}/{len(evaluation_rows)} ({accuracy:.1%})")
print("주의: 이 값은 12개 교육용 예시 시나리오에 한정된 결과이며, 실제 AML/STR 판정이나 운영 정확도로 일반화해서는 안 됩니다.")

✓ [normal_flow] 정상 · 기본 주문: 기대=approve, 추천=approve | score=0.900, margin=0.007
✓ [normal_flow] 정상 · 안정적 반복 구매: 기대=approve, 추천=approve | score=0.926, margin=0.018
✓ [velocity] 관찰 · 빈번한 주문: 기대=manual-review, 추천=manual-review | score=0.908, margin=0.005
✗ [account_device_delivery_linkage] 관찰 · 기기·배송지 연계: 기대=manual-review, 추천=decline | score=0.894, margin=0.007
✗ [payment_irregularity] 관찰 · 결제수단 변경: 기대=manual-review, 추천=decline | score=0.896, margin=0.002
✗ [recipient_contact_anomaly] 관찰 · 수취인·연락처 변경: 기대=manual-review, 추천=decline | score=0.896, margin=0.001
✓ [behavior_inconsistency] 관찰 · 빈도·간격 불일치: 기대=manual-review, 추천=manual-review | score=0.917, margin=0.009
✓ [limit_evasion] 고위험 · 한도 회피: 기대=decline, 추천=decline | score=0.926, margin=0.025
✓ [multi_account_linkage] 고위험 · 다계정 연계: 기대=decline, 추천=decline | score=0.908, margin=0.012
✓ [payment_failure_churn] 고위험 · 결제 실패 누적: 기대=decline, 추천=decline | score=0.906, margin=0.010
✓ [automation_velocity] 고위험 · 자동화 초고빈도: 기대=decline, 추천=decline | sco

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 5</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">작은 라벨 평가셋으로 점검하기</h2>
<p style="margin: 7px 0;">아래 목록에 담당자가 합의한 사례와 기대 라벨을 추가하세요. 이 평가는 결과를 바꾸지 않고, 같은 공개 API의 추천 정확도만 계산합니다.</p>
  </div>
</div>


In [8]:
my_labeled_cases = [
    # ("사례 이름", "고객 행동 원문", "approve"),
    # ("사례 이름", "고객 행동 원문", "manual-review"),
    # ("사례 이름", "고객 행동 원문", "decline"),
]

my_rows = []
for name, case_text, expected in my_labeled_cases:
    outcome = run_decision_recommendation(case_text, settings=settings)
    my_rows.append((name, expected, outcome["recommendation"]))

if my_rows:
    my_correct = sum(expected == predicted for _, expected, predicted in my_rows)
    print(f"내 평가 정확도: {my_correct}/{len(my_rows)} ({my_correct / len(my_rows):.1%})")
    for name, expected, predicted in my_rows:
        print(f"{name}: 기대={expected}, 추천={predicted}")
else:
    print("my_labeled_cases에 라벨 사례를 추가한 뒤 실행하세요.")

my_labeled_cases에 라벨 사례를 추가한 뒤 실행하세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">YAML 후보 문서 A/B 평가 설계</h2>
<p style="margin: 7px 0;">라벨 사례를 6개 이상 추가하고 YAML 후보 문서의 표현을 한 가지씩 바꿔 비교하세요. 각 버전에서 정확도와 오분류 사례를 기록합니다. 고객 입력 원문과 확인 가능한 사실만 query에 담으며, 이 모듈은 승인·반려 같은 상태 변경을 수행하지 않습니다.</p>
  </div>
</div>


In [9]:
# 실습 5의 my_labeled_cases에 사례를 추가한 뒤 다시 실행하세요.
# 고객 입력 원문·YAML 후보·모델 버전을 함께 기록해 A/B 결과를 비교하세요.